In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

/var/data/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# 1. MLP Architecture Definition
# ==========================================
class WinoGrandeMLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) for binary classification of WinoGrande options.
    It takes the concatenated embeddings of a sentence and a candidate word,
    and outputs a probability indicating if the word is the correct answer.
    """
    def __init__(self, embedding_dim=384, hidden_dim=256):
        super(WinoGrandeMLP, self).__init__()
        
        # The input dimension is doubled because we concatenate sentence and word embeddings.
        # e.g., 'all-MiniLM-L6-v2' outputs 384-dimensional vectors -> 384 * 2 = 768.
        input_dim = embedding_dim * 2
        
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # Dropout is used to randomly zero out elements to prevent overfitting during training
            nn.Dropout(0.2), 
            nn.Linear(hidden_dim, int(hidden_dim / 2)),
            nn.ReLU(),
            # Final layer reduces dimension to 1 for binary classification
            nn.Linear(int(hidden_dim / 2), 1),
            # Sigmoid activation squeezes the output between 0 and 1 (interpretable as a probability)
            nn.Sigmoid() 
        )

    def forward(self, sentence_emb, word_emb):
        # Concatenate sentence and word embeddings along the feature dimension (dim=1)
        # Shape: (batch_size, input_dim)
        x = torch.cat((sentence_emb, word_emb), dim=1)
        
        # Pass through the network and remove the extra sequence dimension using squeeze(1)
        # Output shape: (batch_size,)
        return self.mlp(x).squeeze(1)

In [3]:
# ==========================================
# 2. Dataset Preparation
# ==========================================
class WinoGrandeDataset(Dataset):
    """
    Custom PyTorch Dataset for WinoGrande.
    Pre-computes sentence embeddings to speed up the training loop.
    """
    def __init__(self, csv_file, embedding_model_name='all-MiniLM-L6-v2'):
        self.df = pd.read_csv(csv_file)
        # Initialize the HuggingFace SentenceTransformer model
        self.embedder = SentenceTransformer(embedding_model_name)
        
        self.samples = []
        self.labels = []
        
        print(f"Extracting embeddings from {csv_file} (this may take a moment)...")
        self._prepare_data()
        
    def _prepare_data(self):
        """
        Parses the dataframe and converts each row into TWO distinct training samples:
        one for Option 1 and one for Option 2. Label is 1.0 if correct, 0.0 if incorrect.
        """
        sentences = self.df['sentence'].tolist()
        opt1 = self.df['option1'].tolist()
        opt2 = self.df['option2'].tolist()
        # Ensure answers are strings and stripped of whitespace for accurate comparison
        answers = self.df['answer'].astype(str).str.strip().tolist()
        
        # Batch encoding is significantly faster than encoding row by row in a loop.
        # convert_to_tensor=True keeps the output on the GPU (if available) or CPU as PyTorch tensors.
        sent_embs = self.embedder.encode(sentences, convert_to_tensor=True)
        opt1_embs = self.embedder.encode(opt1, convert_to_tensor=True)
        opt2_embs = self.embedder.encode(opt2, convert_to_tensor=True)
        
        # Build the final samples and labels lists
        for i in range(len(self.df)):
            ans = answers[i]
            
            # Sample 1: Sentence + Option 1
            self.samples.append((sent_embs[i], opt1_embs[i]))
            self.labels.append(1.0 if ans == '1' else 0.0)
            
            # Sample 2: Sentence + Option 2
            self.samples.append((sent_embs[i], opt2_embs[i]))
            self.labels.append(1.0 if ans == '2' else 0.0)

    def __len__(self):
        # Required method: returns the total number of individual samples
        return len(self.samples)

    def __getitem__(self, idx):
        # Required method: returns a single sample (features and label) at the given index
        sent_emb, word_emb = self.samples[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return sent_emb, word_emb, label

In [4]:
# ==========================================
# 3. Training Loop
# ==========================================
def train_model(model, train_loader, epochs=5, lr=1e-3):
    """
    Trains the MLP model using Binary Cross Entropy Loss and the Adam optimizer.
    """
    # BCELoss requires the output to be probabilities (between 0 and 1), hence the Sigmoid in the model
    criterion = nn.BCELoss() 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Set the model to training mode (enables Dropout and BatchNorm if used)
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        for sent_emb, word_emb, labels in train_loader:
            # 1. Clear gradients from the previous iteration
            optimizer.zero_grad()
            
            # 2. Forward pass: compute predicted probabilities
            predictions = model(sent_emb, word_emb)
            
            # 3. Compute the loss between predictions and actual labels
            loss = criterion(predictions, labels)
            
            # 4. Backward pass: compute gradients for all parameters
            loss.backward()
            
            # 5. Update model weights based on gradients
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Average Loss: {avg_loss:.4f}")

In [5]:
# ==========================================
# 4. Evaluation Loop
# ==========================================
def evaluate_model(model, df_val, embedder):
    """
    Evaluates the model on a validation or test set.
    For each sentence, it computes the probability for both options and 
    predicts the option with the highest probability.
    """
    # Set the model to evaluation mode (disables Dropout for deterministic behavior)
    model.eval()
    correct = 0
    total = len(df_val)
    
    sentences = df_val['sentence'].tolist()
    opt1 = df_val['option1'].tolist()
    opt2 = df_val['option2'].tolist()
    answers = df_val['answer'].astype(str).str.strip().tolist()
    
    # Disable gradient calculation to save memory and computation during inference
    with torch.no_grad(): 
        # Batch encode the validation set
        sent_embs = embedder.encode(sentences, convert_to_tensor=True)
        opt1_embs = embedder.encode(opt1, convert_to_tensor=True)
        opt2_embs = embedder.encode(opt2, convert_to_tensor=True)
        
        for i in range(total):
            # unsqueeze(0) adds a batch dimension of size 1, required by the model (shape: 1, embedding_dim)
            prob1 = model(sent_embs[i].unsqueeze(0), opt1_embs[i].unsqueeze(0)).item()
            prob2 = model(sent_embs[i].unsqueeze(0), opt2_embs[i].unsqueeze(0)).item()
            
            # Prediction logic: Select the option that yielded the highest confidence score
            predicted_ans = '1' if prob1 > prob2 else '2'
            
            if predicted_ans == answers[i]:
                correct += 1
                
    accuracy = correct / total
    print(f"Accuracy: {accuracy*100:.2f}% ({correct}/{total})")
    return accuracy

In [7]:
# ==========================================
# 5. Main Execution
# ==========================================
if __name__ == "__main__":
    # --- Hyperparameters ---
    EMBEDDING_DIM = 384
    BATCH_SIZE = 32
    EPOCHS = 8
    
    # Initialize the architecture
    model = WinoGrandeMLP(embedding_dim=EMBEDDING_DIM)
    
    # 1. Load data and prepare training loader
    # Make sure 'winogrande_train.csv' is in the working directory
    train_dataset = WinoGrandeDataset("winogrande_train.csv")
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # 2. Start training process
    print("\n--- Starting Training ---")
    train_model(model, train_loader, epochs=EPOCHS)
    
    # 3. Run evaluation
    print("\n--- Evaluating on Validation Data ---")
    val_df = pd.read_csv("winogrande_val.csv")
    
    # Pass the already instantiated embedder to avoid loading the model weights twice into memory
    evaluate_model(model, val_df, train_dataset.embedder)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 882.01it/s]


Extracting embeddings from winogrande_train.csv (this may take a moment)...

--- Starting Training ---
Epoch 1/8 | Average Loss: 0.6934
Epoch 2/8 | Average Loss: 0.6932
Epoch 3/8 | Average Loss: 0.6933
Epoch 4/8 | Average Loss: 0.6932
Epoch 5/8 | Average Loss: 0.6932
Epoch 6/8 | Average Loss: 0.6932
Epoch 7/8 | Average Loss: 0.6932
Epoch 8/8 | Average Loss: 0.6932

--- Evaluating on Validation Data ---
Accuracy: 50.04% (634/1267)
